In [3]:
import pandas as pd
import os

seasons = [
    "E0", "E0 (1)", "E0 (2)", "E0 (3)", "E0 (4)",
    "E0 (5)", "E0 (6)", "E0 (7)", "E0 (8)", "E0 (9)"
]

all_seasons = []

for season in seasons:
    df = pd.read_csv(f"../data/{season}.csv", encoding="latin1")

    df["Season"] = season

    all_seasons.append(df)

df = pd.concat(all_seasons, ignore_index=True)

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

print(df.shape)
print(df.head())
print(df["Season"].value_counts().sort_index())
print(df["Season"].nunique())

(3800, 155)
  ï»¿Div       Date Time     HomeTeam        AwayTeam  FTHG  FTAG FTR  HTHG  \
0    NaN 2015-08-08  NaN  Bournemouth     Aston Villa     0     1   A     0   
1    NaN 2015-08-08  NaN      Norwich  Crystal Palace     1     3   A     0   
2    NaN 2015-08-08  NaN   Man United       Tottenham     1     0   H     1   
3    NaN 2015-08-08  NaN      Chelsea         Swansea     2     2   D     2   
4    NaN 2015-08-08  NaN      Everton         Watford     2     2   D     0   

   HTAG  ... BbAv<2.5  BbAH  BbAHh  BbMxAHH  BbAvAHH  BbMxAHA  BbAvAHA   LBH  \
0     0  ...     1.79  26.0   -0.5     1.98     1.93     1.99     1.92  2.05   
1     1  ...     1.67  27.0    0.0     1.83     1.78     2.17     2.08  2.62   
2     0  ...     2.01  26.0   -1.0     2.20     2.09     1.82     1.78  1.67   
3     1  ...     1.99  27.0   -1.5     2.24     2.16     1.80     1.73  1.40   
4     1  ...     1.96  26.0   -1.0     2.28     2.18     1.76     1.71  1.75   

   LBD   LBA  
0  3.3   4.0  
1 

We now have all 10 seasons in a single dataframe. Let's find the attacking and defensive strengths for the 10 best teams (percentage above/below league average).

In [4]:
league_home_avg = df["FTHG"].mean()
league_away_avg = df["FTAG"].mean()

print("Average home goals:", league_home_avg)
print("Average away goals:", league_away_avg)

teams = sorted(set(df["HomeTeam"]) | set(df["AwayTeam"]))

team_stats = []

for team in teams:

    home_matches = df[df["HomeTeam"] == team]
    away_matches = df[df["AwayTeam"] == team]

    goals_scored = (
        home_matches["FTHG"].sum() +
        away_matches["FTAG"].sum()
    )

    goals_conceded = (
        home_matches["FTAG"].sum() +
        away_matches["FTHG"].sum()
    )

    matches_played = len(home_matches) + len(away_matches)

    avg_scored = goals_scored / matches_played
    avg_conceded = goals_conceded / matches_played

    attack_strength = avg_scored / ((league_home_avg + league_away_avg) / 2)
    defence_strength = avg_conceded / ((league_home_avg + league_away_avg) / 2)

    team_stats.append({
        "Team": team,
        "Matches": matches_played,
        "Goals Scored": goals_scored,
        "Goals Conceded": goals_conceded,
        "Attack Strength": attack_strength,
        "Defence Strength": defence_strength
    })

team_stats = pd.DataFrame(team_stats)

print(team_stats.sort_values("Attack Strength", ascending=False).head(10))

Average home goals: 1.551842105263158
Average away goals: 1.2781578947368422
          Team  Matches  Goals Scored  Goals Conceded  Attack Strength  \
18    Man City      380           898             334         1.670076   
16   Liverpool      380           808             382         1.502697   
0      Arsenal      380           709             423         1.318579   
29   Tottenham      380           702             457         1.305561   
7      Chelsea      380           651             439         1.210712   
19  Man United      380           591             438         1.099126   
3    Brentford      152           228             224         1.060071   
15   Leicester      342           504             505         1.041473   
32    West Ham      380           531             589         0.987540   
21   Newcastle      342           474             484         0.979481   

    Defence Strength  
18          0.621164  
16          0.710433  
0           0.786684  
29          0.84

Our next goal is to predict the expected goals in a fixture.

In [5]:
def predict_expected_goals(home_team, away_team):

    home_attack = team_stats.loc[
        team_stats["Team"] == home_team, "Attack Strength"
    ].iloc[0]

    home_defence = team_stats.loc[
        team_stats["Team"] == home_team, "Defence Strength"
    ].iloc[0]

    away_attack = team_stats.loc[
        team_stats["Team"] == away_team, "Attack Strength"
    ].iloc[0]

    away_defence = team_stats.loc[
        team_stats["Team"] == away_team, "Defence Strength"
    ].iloc[0]

    expected_home = league_home_avg * home_attack * away_defence
    expected_away = league_away_avg * away_attack * home_defence

    return expected_home, expected_away


print(predict_expected_goals("Man City", "Arsenal"))

(np.float64(2.0388447735987603), np.float64(1.0468805615146939))


We can now turn these expected goals into probabilities using the Poisson distribution.

In [6]:
from scipy.stats import poisson
import numpy as np

home_lambda, away_lambda = predict_expected_goals(
    "Man City", "Arsenal"
)

goals = np.arange(0, 11)

home_probs = poisson.pmf(goals, home_lambda)
away_probs = poisson.pmf(goals, away_lambda)

score_matrix = np.outer(home_probs, away_probs)

print(np.round(score_matrix, 3))

home_win = np.tril(score_matrix, -1).sum()
draw = np.trace(score_matrix)
away_win = np.triu(score_matrix, 1).sum()

print(f"Home win: {home_win:.2%}")
print(f"Draw: {draw:.2%}")
print(f"Away win: {away_win:.2%}")

[[0.046 0.048 0.025 0.009 0.002 0.    0.    0.    0.    0.    0.   ]
 [0.093 0.098 0.051 0.018 0.005 0.001 0.    0.    0.    0.    0.   ]
 [0.095 0.099 0.052 0.018 0.005 0.001 0.    0.    0.    0.    0.   ]
 [0.065 0.068 0.035 0.012 0.003 0.001 0.    0.    0.    0.    0.   ]
 [0.033 0.034 0.018 0.006 0.002 0.    0.    0.    0.    0.    0.   ]
 [0.013 0.014 0.007 0.003 0.001 0.    0.    0.    0.    0.    0.   ]
 [0.005 0.005 0.002 0.001 0.    0.    0.    0.    0.    0.    0.   ]
 [0.001 0.001 0.001 0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]]
Home win: 60.28%
Draw: 20.94%
Away win: 18.78%


One important thing to recognise is we are using data from 10 seasons, so if we wanted to predict a match in one of these seasons, it already knows how every team performed in the later seasons. So, we should implement a version that only looks at the past.

In [7]:
def strengths_before(date, matches):
    past = matches[matches["Date"] < date]

    league_home_avg = past["FTHG"].mean()
    league_away_avg = past["FTAG"].mean()
    avg_team = (league_home_avg + league_away_avg) / 2

    rows = []
    teams = sorted(set(past["HomeTeam"]) | set(past["AwayTeam"]))
    for team in teams:
        home_matches = past[past["HomeTeam"] == team]
        away_matches = past[past["AwayTeam"] == team]

        scored = home_matches["FTHG"].sum() + away_matches["FTAG"].sum()
        conceded = home_matches["FTAG"].sum() + away_matches["FTHG"].sum()
        played = len(home_matches) + len(away_matches)

        rows.append({
            "Team": team,
            "Matches": played,
            "Attack": (scored / played) / avg_team,
            "Defence": (conceded / played) / avg_team,
        })

    return pd.DataFrame(rows).set_index("Team")

Next, I want to also introduce recent performances as a factor which influences our probabilities. As I believe this is extremely important as many teams may not have historically performed in the past, therefore they will be vastly underestimated despite great improvement in recent times.

In [8]:
def recent_form(team, date, matches, n=5, venue=None):
    past = matches[matches["Date"] < date]         

    if venue == "home":                             
        games = past[past["HomeTeam"] == team].tail(n)
        scored, conceded = games["FTHG"].to_numpy(), games["FTAG"].to_numpy()
    elif venue == "away":                          
        games = past[past["AwayTeam"] == team].tail(n)
        scored, conceded = games["FTAG"].to_numpy(), games["FTHG"].to_numpy()
    else:                                          
        games = past[(past["HomeTeam"] == team) | (past["AwayTeam"] == team)].tail(n)
        is_home = (games["HomeTeam"] == team).to_numpy()
        scored = np.where(is_home, games["FTHG"], games["FTAG"])
        conceded = np.where(is_home, games["FTAG"], games["FTHG"])

    if len(games) == 0:
        return {"games": 0, "goals_scored": np.nan, "goals_conceded": np.nan, "points": np.nan}

    points = np.where(scored > conceded, 3, np.where(scored == conceded, 1, 0))
    return {"games": len(games),
            "goals_scored": scored.mean(),
            "goals_conceded": conceded.mean(),
            "points": points.mean()}

We can now implement the idea of recent form and team strength into a prediction model.

In [9]:
def predict_match(home_team, away_team, date, matches,
                  form_weight=0.3, n_form=5, venue_form=False, max_goals=10):
    strengths = strengths_before(date, matches)
    past = matches[matches["Date"] < date]
    league_home_avg = past["FTHG"].mean()
    league_away_avg = past["FTAG"].mean()
    avg_team = (league_home_avg + league_away_avg) / 2

    def blended(team, venue):
        attack = strengths.loc[team, "Attack"]
        defence = strengths.loc[team, "Defence"]
        form = recent_form(team, date, matches, n_form, venue if venue_form else None)

        if form["games"] < n_form:                  
            return attack, defence

        recent_attack = form["goals_scored"] / avg_team      
        recent_defence = form["goals_conceded"] / avg_team  
        return ((1 - form_weight) * attack + form_weight * recent_attack,
                (1 - form_weight) * defence + form_weight * recent_defence)

    attack_h, defence_h = blended(home_team, "home")
    attack_a, defence_a = blended(away_team, "away")

    expected_home = league_home_avg * attack_h * defence_a
    expected_away = league_away_avg * attack_a * defence_h

    goals = np.arange(0, max_goals + 1)
    score_matrix = np.outer(poisson.pmf(goals, expected_home),
                            poisson.pmf(goals, expected_away))

    home_win = np.tril(score_matrix, -1).sum()
    draw = np.trace(score_matrix)
    away_win = np.triu(score_matrix, 1).sum()
    return expected_home, expected_away, home_win, draw, away_win

Let's implement a walk forward model where we predict every match in a season, using only what we know before it. We'll use 23/24 as our validation season.

In [10]:
val_matches = df[(df["Date"] >= "2023-08-01") & (df["Date"] < "2024-08-01")]

results = []
skipped = 0

for i, match in val_matches.iterrows():
    try:
        xg_h, xg_a, p_h, p_d, p_a = predict_match(
            match["HomeTeam"], match["AwayTeam"], match["Date"], df, form_weight=0.3
        )
    except KeyError:        
        skipped += 1
        continue

    results.append({
        "Date": match["Date"],
        "HomeTeam": match["HomeTeam"],
        "AwayTeam": match["AwayTeam"],
        "FTR": match["FTR"],    
        "pH": p_h, "pD": p_d, "pA": p_a,
    })

predictions = pd.DataFrame(results)

print("Matches in season:", len(val_matches))
print("Predicted:", len(predictions), "| Skipped:", skipped)
print(predictions.head())

Matches in season: 380
Predicted: 379 | Skipped: 1
        Date          HomeTeam        AwayTeam FTR        pH        pD  \
0 2023-08-11           Burnley        Man City   A  0.099260  0.190983   
1 2023-08-12       Bournemouth        West Ham   D  0.350594  0.252773   
2 2023-08-12           Everton          Fulham   A  0.510317  0.230245   
3 2023-08-12  Sheffield United  Crystal Palace   A  0.231551  0.306344   
4 2023-08-12         Newcastle     Aston Villa   H  0.456106  0.272042   

         pA  
0  0.709747  
1  0.396633  
2  0.259435  
3  0.462105  
4  0.271851  


Now, we score our predictions with log loss, so it rewards confident correct predictions and punishes confident wrong ones heavily.

In [11]:
def log_loss(probs, results):
    p_actual = np.where(results == "H", probs["pH"],
                np.where(results == "D", probs["pD"], probs["pA"]))
    return -np.mean(np.log(p_actual))

train = df[df["Date"] < "2023-08-01"]
base = train["FTR"].value_counts(normalize=True)
baseline = pd.DataFrame({"pH": base["H"], "pD": base["D"], "pA": base["A"]},
                        index=predictions.index)

print("Your model:      ", round(log_loss(predictions, predictions["FTR"]), 4))
print("Base rates:      ", round(log_loss(baseline, predictions["FTR"]), 4))
print("Know-nothing 1/3:", round(np.log(3), 4))

Your model:       0.9287
Base rates:       1.055
Know-nothing 1/3: 1.0986


How much should recent form influence a team's estimated strength? How many previous matches should be used to calculate recent form?

In [12]:
val_matches = df[
    (df["Date"] >= "2023-08-01") &
    (df["Date"] < "2024-08-01")
].copy()
weights = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
windows = [3, 5, 8, 10]

results = []

for weight in weights:
    for window in windows:

        actual = []
        predicted = []

        for _, match in val_matches.iterrows():
            try:
                _, _, pH, pD, pA = predict_match(
                    match["HomeTeam"],
                    match["AwayTeam"],
                    match["Date"],
                    df,
                    form_weight=weight,
                    n_form=window
                )

                actual.append(match["FTR"])
                predicted.append([pH, pD, pA])

            except KeyError:
                continue

        loss = log_loss(
            pd.DataFrame(predicted, columns=["pH", "pD", "pA"]),
            np.array(actual)
        )

        results.append({
            "Weight": weight,
            "Window": window,
            "Log Loss": loss
        })

results_df = pd.DataFrame(results).sort_values("Log Loss")

print(results_df)

    Weight  Window  Log Loss
9      0.2       5  0.927123
13     0.3       5  0.928731
5      0.1       5  0.929862
10     0.2       8  0.930131
11     0.2      10  0.930701
14     0.3       8  0.931050
15     0.3      10  0.931184
6      0.1       8  0.932131
7      0.1      10  0.932670
4      0.1       3  0.932911
19     0.4      10  0.934050
17     0.4       5  0.934542
18     0.4       8  0.934795
8      0.2       3  0.936367
0      0.0       3  0.937189
3      0.0      10  0.937189
2      0.0       8  0.937189
1      0.0       5  0.937189
23     0.5      10  0.939254
22     0.5       8  0.941313
21     0.5       5  0.944497
27     0.6      10  0.946774
12     0.3       3  0.947241
26     0.6       8  0.950589
25     0.6       5  0.958625
16     0.4       3  0.965455
20     0.5       3  0.991157
24     0.6       3  1.024777


No-form model has a log loss of 0.937189 and the best model achieves 0.927123, that's an improvement of approximately 1.07% in log loss. This suggests recent form contains useful predictive information beyond overall team strengths.
Results suggest that recent form should receive approximately 20% weight, while historical attacking and defensive strengths receive 80%. This is intuitive: recent performance matters, but five matches can be a relatively noisy sample.
The five-match window consistently performs well across different weights. Interestingly, increasing the window to eight or ten matches doesn't improve the best result.

In [13]:
results = []

for venue in [False, True]:

    actual = []
    predicted = []

    for _, match in val_matches.iterrows():
        try:
            _, _, pH, pD, pA = predict_match(
                match["HomeTeam"],
                match["AwayTeam"],
                match["Date"],
                df,
                form_weight=0.2,
                n_form=5,
                venue_form=venue
            )

            actual.append(match["FTR"])
            predicted.append([pH, pD, pA])

        except KeyError:
            continue

    loss = log_loss(
        pd.DataFrame(predicted, columns=["pH", "pD", "pA"]),
        np.array(actual)
    )

    results.append({
        "Venue-specific form": venue,
        "Log Loss": loss,
        "Matches": len(actual)
    })

print(pd.DataFrame(results))

   Venue-specific form  Log Loss  Matches
0                False  0.927123      379
1                 True  0.934778      379


Before making the model more complicated, we need to know whether it actually beats a simple prediction strategy. Calculate the log loss of a baseline that predicts every match using the historical proportions of home wins, draws and away wins.

In [14]:
train = df[df["Date"] < "2023-08-01"]

base = train["FTR"].value_counts(normalize=True)

baseline_probs = pd.DataFrame({
    "pH": base["H"],
    "pD": base["D"],
    "pA": base["A"]
}, index=predictions.index)

baseline_loss = log_loss(
    baseline_probs,
    predictions["FTR"].to_numpy()
)

print("Baseline log loss:", baseline_loss)
print("Poisson model log loss:", log_loss(
    predictions[["pH", "pD", "pA"]],
    predictions["FTR"].to_numpy()
))

Baseline log loss: 1.0550019247217024
Poisson model log loss: 0.9287306823054302


Next, let's investigate calibration. A model can have good log loss but still systematically overestimate or underestimate certain probabilities.

In [15]:
predictions["HomeProbabilityBin"] = pd.cut(
    predictions["pH"],
    bins=np.arange(0, 1.1, 0.1)
)

calibration = predictions.groupby(
    "HomeProbabilityBin",
    observed=False
).agg(
    MeanPredicted=("pH", "mean"),
    ActualHomeWinRate=("FTR", lambda x: (x == "H").mean()),
    Matches=("FTR", "count")
)

print(calibration)

                    MeanPredicted  ActualHomeWinRate  Matches
HomeProbabilityBin                                           
(0.0, 0.1]               0.067810           0.083333       12
(0.1, 0.2]               0.150006           0.142857       35
(0.2, 0.3]               0.259511           0.258621       58
(0.3, 0.4]               0.351691           0.397059       68
(0.4, 0.5]               0.449532           0.516667       60
(0.5, 0.6]               0.542876           0.517857       56
(0.6, 0.7]               0.653633           0.638889       36
(0.7, 0.8]               0.746950           0.677419       31
(0.8, 0.9]               0.845590           0.947368       19
(0.9, 1.0]               0.915518           1.000000        4


The model is not simply predicting home wins too frequently or too infrequently across the board. Its errors vary depending on how confident it is. 